# Pandas Lecture 3

This lecture covers:
- **Section: Advanced filtering expressions** (several conditions with `&`, `|`, `~`)
- **Section: Row and column subsetting (slicing) together using `iloc` and `loc`**
- **Section: Introduction to GroupBy** (grouping rows and summarizing each group)

We will use the company dataset from Lectures 1 and 2. This time we read the **full dataset** from a CSV file.

> Note: the rest of **GroupBy** and **missing values** will be covered in **Lecture 4**.

## Setup: Load the company dataset and do a quick check

Up to Lecture 2, the company data was typed inside the notebook. Starting from this lecture, we **read the data from a CSV file** with `pd.read_csv()`.

The file `company_monthly.csv` has the **full company dataset**: **30 companies × 72 months** (January 2020 to December 2025) = **2,160 rows**.

The first 90 rows are the rows we used in Lectures 1 and 2.

The data is **simulated**. These are **not real companies**.

✅ **Before you run the next cells**

1. **Download** the file `company_monthly.csv` from the class GitHub site and **upload** it to your Google Drive.
2. **Change the path** in `pd.read_csv(...)`. The path is different for everyone, so make sure to put the right path for your own case.

✅ **Tip:** In case you forgot it, see my lecture video on "how to mount Google Drive and read the CSV file" (the homework assignment two weeks ago).

In [1]:
# Mount your Google Drive
# (in case you forgot it, see my lecture video on "how to mount Google Drive and read the CSV file",
#  the homework assignment two weeks ago)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# Read the CSV file into a DataFrame
# The path is different for everyone --> make sure to put the right path for your own case
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/company_monthly.csv")

df.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
0,ALDG-2020-01,ALDG,Aldgate Machinery,Industrials,Mid,Yes,2020-01-31,2020,1,41.51,-4.35,15.77,4639.6,255.7,13.78,-1.38
1,AVLO-2020-01,AVLO,Arvelo Systems,Technology,Large,No,2020-01-31,2020,1,186.66,-2.88,48.54,78656.9,891.7,207.10,-1.38
2,BRKF-2020-01,BRKF,Brackley Freight,Industrials,Mid,Yes,2020-01-31,2020,1,41.98,-3.63,22.27,6809.8,338.9,20.01,-1.38
3,CLDW-2020-01,CLDW,Caldwyn Aerospace,Industrials,Large,Yes,2020-01-31,2020,1,218.64,-8.75,23.04,36722.6,2106.1,167.71,-1.38
4,CRVN-2020-01,CRVN,Corvane Software,Technology,Large,Yes,2020-01-31,2020,1,285.48,-4.60,15.74,49438.4,674.6,121.64,-1.38


In [3]:
# (rows, columns)
df.shape

(2160, 16)

# Section: Advanced filtering expressions

In Lecture 2, a filter had **one** condition, for example `df[df["Price"] >= 100]`. Now we combine **several** conditions.

- Use `&` for AND, `|` for OR, `~` for NOT
- Put **each condition in parentheses**

**Typical syntax:**

```
df[ (condition1) & (condition2) ]     AND: both must be True
df[ (condition1) | (condition2) ]     OR: at least one must be True
df[ ~(condition) ]                    NOT: flips True and False
```

### Boolean Operator Reference Tables

Below are the truth tables for the three logical operators used in Pandas row filtering.

**AND (`&`)**: both conditions must be `True`

| A | B | A & B |
|---|---|-------|
| True | True | **True** |
| True | False | False |
| False | True | False |
| False | False | False |

**OR (`|`)**: at least one condition must be `True`

| A | B | A \| B |
|---|---|--------|
| True | True | **True** |
| True | False | **True** |
| False | True | **True** |
| False | False | False |

**NOT (`~`)**: flips `True` and `False`

| A | ~A |
|---|-----|
| True | False |
| False | **True** |

## AND with `&`

**Q: Which rows are in the Technology sector AND have a `Price` of at least 100?**

In [4]:
df["Sector"] == "Technology" & df["Price"] >= 100   # Error!!! --> put each condition in ( )
# See the solution below

TypeError: Cannot perform 'rand_' with a dtyped [float64] array and scalar of type [bool]

In [5]:
(df["Sector"] == "Technology") & (df["Price"] >= 100)   # boolean expression: True only when BOTH are True

0       False
1        True
2       False
3       False
4        True
        ...  
2155    False
2156    False
2157    False
2158    False
2159    False
Length: 2160, dtype: bool

In [6]:
# Technology AND Price of at least 100
tech_100 = df[(df["Sector"] == "Technology") & (df["Price"] >= 100)]
tech_100.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
1,AVLO-2020-01,AVLO,Arvelo Systems,Technology,Large,No,2020-01-31,2020,1,186.66,-2.88,48.54,78656.9,891.7,207.10,-1.38
4,CRVN-2020-01,CRVN,Corvane Software,Technology,Large,Yes,2020-01-31,2020,1,285.48,-4.60,15.74,49438.4,674.6,121.64,-1.38
5,DLMR-2020-01,DLMR,Delmora Semiconductor,Technology,Large,Yes,2020-01-31,2020,1,272.26,-2.90,12.18,27190.1,355.1,68.98,-1.38
31,AVLO-2020-02,AVLO,Arvelo Systems,Technology,Large,No,2020-02-29,2020,2,197.84,5.99,49.57,83368.1,911.1,192.73,6.47
34,CRVN-2020-02,CRVN,Corvane Software,Technology,Large,Yes,2020-02-29,2020,2,304.22,6.56,18.74,52683.8,658.8,122.40,6.47


In [7]:
# How many rows are in the Technology sector AND have a Price of at least 100?
tech_100.shape

(316, 16)

In [8]:
len(tech_100)

316

## OR with `|`

**Q: Which rows are in the Technology sector OR have a `Price` of at least 100 (either condition is enough)?**

In [9]:
# Technology OR Price of at least 100 (either condition is enough)
tech_or_100 = df[(df["Sector"] == "Technology") | (df["Price"] >= 100)]
tech_or_100.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
1,AVLO-2020-01,AVLO,Arvelo Systems,Technology,Large,No,2020-01-31,2020,1,186.66,-2.88,48.54,78656.9,891.7,207.10,-1.38
3,CLDW-2020-01,CLDW,Caldwyn Aerospace,Industrials,Large,Yes,2020-01-31,2020,1,218.64,-8.75,23.04,36722.6,2106.1,167.71,-1.38
4,CRVN-2020-01,CRVN,Corvane Software,Technology,Large,Yes,2020-01-31,2020,1,285.48,-4.60,15.74,49438.4,674.6,121.64,-1.38
5,DLMR-2020-01,DLMR,Delmora Semiconductor,Technology,Large,Yes,2020-01-31,2020,1,272.26,-2.90,12.18,27190.1,355.1,68.98,-1.38
7,ELVR-2020-01,ELVR,Elvora Cloud,Technology,Mid,No,2020-01-31,2020,1,70.90,-3.10,16.66,9177.9,144.1,15.58,-1.38


In [10]:
# How many --> OR keeps more rows than AND (316 rows)
tech_or_100.shape

(1269, 16)

**Q: Which rows are in the Energy sector OR the Financials sector?**

In [11]:
# Sector is Energy OR Financials (two conditions on the same column)
energy_fin = df[(df["Sector"] == "Energy") | (df["Sector"] == "Financials")]
energy_fin.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
8,ESTF-2020-01,ESTF,Eastfell Energy,Energy,Large,Yes,2020-01-31,2020,1,82.58,4.22,99.49,125850.8,7680.6,937.89,-1.38
10,FRLW-2020-01,FRLW,Fairlow Petroleum,Energy,Mid,Yes,2020-01-31,2020,1,70.68,2.52,5.21,4272.8,376.6,30.23,-1.38
11,GRSL-2020-01,GRSL,Garrick Solar,Energy,Small,No,2020-01-31,2020,1,33.99,3.12,3.88,413.2,34.2,-2.98,-1.38
13,IRVB-2020-01,IRVB,Ironvale Bank,Financials,Large,Yes,2020-01-31,2020,1,93.30,-0.63,59.51,51223.6,1593.6,327.66,-1.38
15,JSMY-2020-01,JSMY,Jessamy Trust,Financials,Mid,Yes,2020-01-31,2020,1,122.02,-1.56,12.21,10495.6,366.7,53.23,-1.38


In [12]:
len(energy_fin)   # 216 Energy rows + 432 Financials rows

648

## Between two values

**Q: Which rows have a `Price` between 50 and 100 (inclusive)?**

(`<=` means less than or equal to.)

In [13]:
# Price between 50 and 100 (inclusive)
# Use two comparisons + & (AND)
mid_price = df[(df["Price"] >= 50) & (df["Price"] <= 100)]
mid_price.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
7,ELVR-2020-01,ELVR,Elvora Cloud,Technology,Mid,No,2020-01-31,2020,1,70.90,-3.10,16.66,9177.9,144.1,15.58,-1.38
8,ESTF-2020-01,ESTF,Eastfell Energy,Energy,Large,Yes,2020-01-31,2020,1,82.58,4.22,99.49,125850.8,7680.6,937.89,-1.38
9,FNNK-2020-01,FNNK,Fennick Data,Technology,Mid,No,2020-01-31,2020,1,80.60,2.57,NaN,3115.4,35.2,5.94,-1.38
10,FRLW-2020-01,FRLW,Fairlow Petroleum,Energy,Mid,Yes,2020-01-31,2020,1,70.68,2.52,5.21,4272.8,376.6,30.23,-1.38
13,IRVB-2020-01,IRVB,Ironvale Bank,Financials,Large,Yes,2020-01-31,2020,1,93.30,-0.63,59.51,51223.6,1593.6,327.66,-1.38


In [14]:
len(mid_price)

516

## NOT with `~`

**Q: Which rows are for companies that do NOT pay a dividend?**

In [15]:
~(df["PaysDividend"] == "Yes")     # ~ flips True and False

0       False
1        True
2       False
3       False
4       False
        ...  
2155    False
2156    False
2157    False
2158     True
2159    False
Name: PaysDividend, Length: 2160, dtype: bool

In [16]:
(df["PaysDividend"] != "Yes")      # != means "not equal to" --> the same result

0       False
1        True
2       False
3       False
4       False
        ...  
2155    False
2156    False
2157    False
2158     True
2159    False
Name: PaysDividend, Length: 2160, dtype: bool

In [17]:
df["PaysDividend"] == "No"         # the same result again (the column has only "Yes" and "No")

0       False
1        True
2       False
3       False
4       False
        ...  
2155    False
2156    False
2157    False
2158     True
2159    False
Name: PaysDividend, Length: 2160, dtype: bool

In [18]:
# NOT condition: companies that do NOT pay a dividend
no_div = df[~(df["PaysDividend"] == "Yes")]
no_div.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
1,AVLO-2020-01,AVLO,Arvelo Systems,Technology,Large,No,2020-01-31,2020,1,186.66,-2.88,48.54,78656.9,891.7,207.10,-1.38
7,ELVR-2020-01,ELVR,Elvora Cloud,Technology,Mid,No,2020-01-31,2020,1,70.90,-3.10,16.66,9177.9,144.1,15.58,-1.38
9,FNNK-2020-01,FNNK,Fennick Data,Technology,Mid,No,2020-01-31,2020,1,80.60,2.57,NaN,3115.4,35.2,5.94,-1.38
11,GRSL-2020-01,GRSL,Garrick Solar,Energy,Small,No,2020-01-31,2020,1,33.99,3.12,3.88,413.2,34.2,-2.98,-1.38
12,HLVN-2020-01,HLVN,Halvern Networks,Technology,Small,No,2020-01-31,2020,1,42.35,6.30,1.50,585.7,9.1,0.26,-1.38


In [19]:
len(no_div)

720

## ✅ Practice (Advanced filtering expressions)

These are a little harder. **Answers are shown right below each question.**

### Q1) AND condition
**Which rows have a `Return` greater than 10 AND a `SizeGroup` of `Large`?**

**Answer**
```python
df[(df["Return"] > 10) & (df["SizeGroup"] == "Large")]     # 48 rows
```

### Q2) OR condition
**Which rows are from `Month` 1 OR `Month` 3?**

**Answer**
```python
df[(df["Month"] == 1) | (df["Month"] == 3)]     # 360 rows
```

### Q3) NOT condition + AND
**Which rows are NOT in the `Technology` sector AND have a `Price` of at least 200?**

**Answer**
```python
df[~(df["Sector"] == "Technology") & (df["Price"] >= 200)]     # 320 rows
```

### Q4) Combining AND + OR (multiple conditions)
**Which rows have a `Price` between 50 and 100 (inclusive) AND are in the `Energy` or `Financials` sector?**

**Answer**
```python
df[
    ((df["Sector"] == "Energy") | (df["Sector"] == "Financials"))
    & (df["Price"] >= 50)
    & (df["Price"] <= 100)
]     # 84 rows
```

In [20]:
# Your turn: try the Advanced filtering practice questions here

# Section: Row and column subsetting (slicing) together using `iloc` and `loc`

So far we have learned:
- **Column subsetting:** `df["col"]` or `df[["col1", "col2"]]` → select columns
- **Row subsetting:** `df[boolean_expression]` → filter rows

But what if you want to select **specific rows AND specific columns at the same time**?

Pandas provides two indexers for this:

| Indexer | Selects by | Syntax |
|---------|-----------|--------|
| `df.iloc[row, col]` | **Integer position** (0, 1, 2, …) | `df.iloc[row_position, col_position]` |
| `df.loc[row, col]` | **Label / index name** | `df.loc[row_label, col_name]` |

**Think of it as:**
- **`iloc`** = **i**nteger **loc**ation (use numbers)
- **`loc`** = **l**abel-based **loc**ation (use index labels and column names)

In Lecture 1, we used `iloc` and `loc` to select **rows only**. Now we add the **column part** after the comma.

## `iloc`: selection by integer position

**Q: What is the value in the 1st row and 1st column of the DataFrame (position [0, 0])?**

In [21]:
df.head()

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
0,ALDG-2020-01,ALDG,Aldgate Machinery,Industrials,Mid,Yes,2020-01-31,2020,1,41.51,-4.35,15.77,4639.6,255.7,13.78,-1.38
1,AVLO-2020-01,AVLO,Arvelo Systems,Technology,Large,No,2020-01-31,2020,1,186.66,-2.88,48.54,78656.9,891.7,207.10,-1.38
2,BRKF-2020-01,BRKF,Brackley Freight,Industrials,Mid,Yes,2020-01-31,2020,1,41.98,-3.63,22.27,6809.8,338.9,20.01,-1.38
3,CLDW-2020-01,CLDW,Caldwyn Aerospace,Industrials,Large,Yes,2020-01-31,2020,1,218.64,-8.75,23.04,36722.6,2106.1,167.71,-1.38
4,CRVN-2020-01,CRVN,Corvane Software,Technology,Large,Yes,2020-01-31,2020,1,285.48,-4.60,15.74,49438.4,674.6,121.64,-1.38


In [22]:
# Single element: row 0, column 0
df.iloc[0, 0]

'ALDG-2020-01'

**Q: What are the first 5 rows and first 3 columns of the DataFrame?**

In [23]:
# First 5 rows, first 3 columns (using slicing)
df.iloc[0:5, 0:3]

,RecordID,Ticker,Company
0,ALDG-2020-01,ALDG,Aldgate Machinery
1,AVLO-2020-01,AVLO,Arvelo Systems
2,BRKF-2020-01,BRKF,Brackley Freight
3,CLDW-2020-01,CLDW,Caldwyn Aerospace
4,CRVN-2020-01,CRVN,Corvane Software


**Q: What are the values in rows 0, 30 and 60 for the columns at positions 2, 9 and 10 (`Company`, `Price` and `Return`)?**

*Tip:* `df.columns` shows the column names in order. Count from 0.

In [24]:
df.columns   # RecordID is 0, Ticker is 1, Company is 2, ... Price is 9, Return is 10

Index(['RecordID', 'Ticker', 'Company', 'Sector', 'SizeGroup', 'PaysDividend',
       'Date', 'Year', 'Month', 'Price', 'Return', 'Volume', 'MarketCap',
       'Revenue', 'NetIncome', 'MarketReturn'],
      dtype='object')

In [25]:
# Specific rows and columns by position (using lists)
df.iloc[[0, 30, 60], [2, 9, 10]]

,Company,Price,Return
0,Aldgate Machinery,41.51,-4.35
30,Aldgate Machinery,46.90,12.98
60,Aldgate Machinery,48.51,3.43


**Q: What are all columns for the first 3 rows?**

*Tip:* Use `:` to mean "all" for either rows or columns.

In [26]:
# All columns for the first 3 rows
df.iloc[0:3, :]

,RecordID,Ticker,Company,Sector,SizeGroup,PaysDividend,Date,Year,Month,Price,Return,Volume,MarketCap,Revenue,NetIncome,MarketReturn
0,ALDG-2020-01,ALDG,Aldgate Machinery,Industrials,Mid,Yes,2020-01-31,2020,1,41.51,-4.35,15.77,4639.6,255.7,13.78,-1.38
1,AVLO-2020-01,AVLO,Arvelo Systems,Technology,Large,No,2020-01-31,2020,1,186.66,-2.88,48.54,78656.9,891.7,207.10,-1.38
2,BRKF-2020-01,BRKF,Brackley Freight,Industrials,Mid,Yes,2020-01-31,2020,1,41.98,-3.63,22.27,6809.8,338.9,20.01,-1.38


## `loc`: selection by label / column name

**Q: What are the `Company`, `Price` and `Return` values for the row with index label 0?**

In [27]:
# Single row by index label, specific columns by name
df.loc[0, ["Company", "Price", "Return"]]

Company    Aldgate Machinery
Price                  41.51
Return                 -4.35
Name: 0, dtype: object

**Q: What are the `Company`, `Price` and `Return` values for rows with index labels 0 through 4?**

*Note:* Unlike `iloc`, `loc` slicing is **inclusive** on both ends: `0:4` returns rows 0, 1, 2, 3, **and** 4.

In [28]:
# Rows 0 to 4 (inclusive!), selected columns by name
df.loc[0:4, ["Company", "Price", "Return"]]

,Company,Price,Return
0,Aldgate Machinery,41.51,-4.35
1,Arvelo Systems,186.66,-2.88
2,Brackley Freight,41.98,-3.63
3,Caldwyn Aerospace,218.64,-8.75
4,Corvane Software,285.48,-4.60


In [29]:
# please write the identical expression using iloc (good exam question)
df.iloc[0:5, [2, 9, 10]]

,Company,Price,Return
0,Aldgate Machinery,41.51,-4.35
1,Arvelo Systems,186.66,-2.88
2,Brackley Freight,41.98,-3.63
3,Caldwyn Aerospace,218.64,-8.75
4,Corvane Software,285.48,-4.60


## Combining boolean filters with `loc`

You can also use a **boolean expression** as the row selector inside `loc` to filter rows AND select specific columns at the same time.

```
df.loc[boolean_expression, [col1, col2, ...]]
```

**Q: For the rows with a negative `Return`, what are the `Company`, `Month` and `Return` values?**

In [30]:
# Boolean filter + column selection using loc
df.loc[df["Return"] < 0, ["Company", "Month", "Return"]].head()

,Company,Month,Return
0,Aldgate Machinery,1,-4.35
1,Arvelo Systems,1,-2.88
2,Brackley Freight,1,-3.63
3,Caldwyn Aerospace,1,-8.75
4,Corvane Software,1,-4.60


**Q: For Technology companies with a `Return` greater than 10, what are the `Company`, `Month`, `Price` and `Return` values?**

In [31]:
# Multiple boolean conditions + column selection
df.loc[(df["Sector"] == "Technology") & (df["Return"] > 10), ["Company", "Month", "Price", "Return"]]

,Company,Month,Price,Return
35,Delmora Semiconductor,2,308.23,13.21
37,Elvora Cloud,2,81.85,15.44
39,Fennick Data,2,93.35,15.82
124,Corvane Software,5,306.19,13.86
125,Delmora Semiconductor,5,345.63,10.22
...,...,...,...,...
1984,Corvane Software,7,255.64,10.33
2079,Fennick Data,10,93.41,10.73
2082,Halvern Networks,10,208.11,30.24
2104,Corvane Software,11,271.25,10.65


## `iloc` vs `loc`: quick comparison

| Feature | `iloc` | `loc` |
|---------|--------|-------|
| Selects by | Integer position (0, 1, 2, …) | Label / column name |
| Slicing end | **Exclusive** (`0:3` → rows 0, 1, 2) | **Inclusive** (`0:3` → rows 0, 1, 2, 3) |
| Column reference | Column number (e.g., `9`) | Column name (e.g., `"Price"`) |
| Boolean filter as row selector | ❌ Not supported | ✅ Supported |

## ✅ Practice (Row and column subsetting with `iloc` and `loc`)

Try these first. **Answers are shown right below each question.**

### Q1) `iloc`: slice a block
**What are the values in rows 10 through 14 and columns at positions 1 through 4?**

**Answer**
```python
df.iloc[10:15, 1:5]
```

### Q2) `loc`: select by name
**What are the `Ticker`, `Sector` and `MarketCap` values for rows with index labels 0 through 9?**

**Answer**
```python
df.loc[0:9, ["Ticker", "Sector", "MarketCap"]]
```

### Q3) `loc` with a boolean filter
**For rows in the `Financials` sector with a `Return` less than or equal to -5, what are the `Company`, `Month`, `Price` and `Return` values?**

**Answer**
```python
df.loc[
    (df["Sector"] == "Financials") & (df["Return"] <= -5),
    ["Company", "Month", "Price", "Return"]
]     # 87 rows
```

### Q4) `iloc`: pick specific rows and columns
**What are the values at rows 5, 35 and 65 for the 1st, 10th and 16th columns (positions 0, 9 and 15)?**

**Answer**
```python
df.iloc[[5, 35, 65], [0, 9, 15]]     # Delmora Semiconductor in January, February and March 2020
```

In [32]:
# Your turn: try the iloc and loc practice questions here

# Section: Introduction to GroupBy (grouping rows and summarizing each group)

## What is `groupby`?

`groupby` follows the **split → apply → combine** idea:

1. **Split** the DataFrame into groups using a column (example: `Sector`)
2. **Apply** a summary function to each group (example: `mean`, `sum`, `count`)
3. **Combine** the results into a summary table

**Typical syntax**
```python
df.groupby("GroupColumn")["TargetColumn"].mean()
```

## Count rows in each group: `size()`

**Question:** How many rows are in each `Sector` group?

In [33]:
df.groupby("Sector").size()

Sector
Consumer       360
Energy         216
Financials     432
Health Care    360
Industrials    288
Technology     504
dtype: int64

✅ **Note**
- `.size()` counts **all rows** in each group.
- One row is one company in one month, so Energy has 216 rows (3 companies × 72 months).

## Group average (mean) of one column

**Question:** What is the average `Return` for each `Sector`?

In [34]:
df.groupby("Sector")["Return"].mean()

Sector
Consumer       0.490639
Energy         0.769352
Financials     0.943843
Health Care    0.199472
Industrials    0.752986
Technology     1.086627
Name: Return, dtype: float64

In [35]:
# Total Revenue by Sector (use sum() instead of mean())
df.groupby("Sector")["Revenue"].sum()

Sector
Consumer       1556775.1
Energy          577140.5
Financials      364605.3
Health Care      97131.1
Industrials     237831.2
Technology      254922.1
Name: Revenue, dtype: float64

## ✅ Practice (Introduction to GroupBy)

Try these first. **Answers are shown right below each question.**

### Q1) Average `Price` by `SizeGroup`

**Answer**
```python
df.groupby("SizeGroup")["Price"].mean()
```

### Q2) How many rows are in each `SizeGroup` group?

**Answer**
```python
df.groupby("SizeGroup").size()     # Large 720, Mid 864, Small 576
```

In [36]:
# Your turn: try the GroupBy practice questions here

## ✅ Summary

- `df[(condition1) & (condition2)]` → AND; `|` → OR; `~` → NOT (put each condition in parentheses)
- `df.iloc[rows, cols]` → select by **position** (the end of a slice is excluded)
- `df.loc[rows, cols]` → select by **label and column name** (the end of a slice is included)
- `df.loc[condition, [cols]]` → filter rows and select columns at the same time
- `df.groupby("Group").size()` → number of rows in each group
- `df.groupby("Group")["col"].mean()` → average of a column in each group (use `.sum()` for the total)